<a href="https://colab.research.google.com/github/vis00037/clinical-triage-nlp-baseline/blob/main/Clinical_Triage_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets accelerate torch scikit-learn

In [2]:
import numpy as np
import pandas as pd
import re
# Set USE_SYNTHETIC = False once you download 'triage.csv.gz' from PhysioNet
USE_SYNTHETIC = True
if USE_SYNTHETIC:
   # Realistic synthetic records matching exact MIMIC-IV-ED triage.csv columns
   data = {
       "chiefcomplaint": [
           "chest pain, diaphoresis, sob", "twisted right ankle, mild pain",
           "slurred speech, left sided weakness", "sore throat, runny nose x 2 days",
           "severe abdominal pain, vomiting blood", "minor laceration left index finger",
           "shortness of breath, history of chf", "medication refill, no acute complaints",
           "syncope, dizziness, palpitations", "rash on arm, itchy, no fever"
       ] * 100,
       "temperature": [98.6, 98.4, 98.9, 99.8, 97.9, 98.5, 98.7, 98.6, 98.2, 98.8] * 100,
       "heartrate": [112, 78, 95, 84, 122, 72, 108, 70, 118, 76] * 100,
       "resprate": [24, 16, 18, 16, 26, 15, 25, 14, 22, 16] * 100,
       "o2sat": [91, 99, 97, 98, 94, 99, 89, 99, 95, 99] * 100,
       "sbp": [88, 124, 178, 118, 92, 120, 155, 122, 95, 119] * 100,
       "dbp": [54, 78, 98, 74, 58, 76, 92, 78, 60, 75] * 100,
       "acuity": [1, 4, 1, 5, 2, 4, 2, 5, 2, 4] * 100
   }
   df = pd.DataFrame(data)
else:
   df = pd.read_csv("triage.csv.gz")
# Drop rows missing chief complaint or ESI acuity label
df = df.dropna(subset=["chiefcomplaint", "acuity"]).copy()
# Binary clinical target: 1 = Critical/High Acuity (ESI 1-2), 0 = Lower Acuity (ESI 3-5)
df["high_acuity"] = (df["acuity"] <= 2).astype(int)
print(f"Dataset shape: {df.shape} | High-Acuity Rate: {df['high_acuity'].mean():.2%}")

Dataset shape: (1000, 9) | High-Acuity Rate: 50.00%


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
# Expand common triage nursing abbreviations
ABBREV_MAP = {
   r"\bsob\b": "shortness of breath",
   r"\bcp\b": "chest pain",
   r"\babd\b": "abdominal",
   r"\bha\b": "headache",
   r"\bchf\b": "congestive heart failure",
   r"\bloc\b": "loss of consciousness",
   r"\bhtn\b": "hypertension"
}
def clean_triage_text(text):
   text = str(text).lower().strip()
   for pattern, replacement in ABBREV_MAP.items():
       text = re.sub(pattern, replacement, text)
   text = re.sub(r"[^a-z0-9\s]", " ", text)
   return re.sub(r"\s+", " ", text).strip()
df["clean_text"] = df["chiefcomplaint"].apply(clean_triage_text)
vital_cols = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp"]
# Stratified split (80% train, 20% test)
train_df, test_df = train_test_split(
   df, test_size=0.2, random_state=42, stratify=df["high_acuity"]
)
# Preprocess structured vitals (fit only on train set to prevent data leakage)
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
X_train_vitals = scaler.fit_transform(imputer.fit_transform(train_df[vital_cols]))
X_test_vitals = scaler.transform(imputer.transform(test_df[vital_cols]))
print("Section 2 complete! Train rows:", len(train_df), "| Test rows:", len(test_df))

Section 2 complete! Train rows: 800 | Test rows: 200


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack, csr_matrix
# Convert symptom text to TF-IDF features (unigrams + bigrams)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_text = tfidf.fit_transform(train_df["clean_text"])
X_test_text = tfidf.transform(test_df["clean_text"])
# Multimodal fusion: concatenate text vectors + structured vital signs
X_train_multi = hstack([X_train_text, csr_matrix(X_train_vitals)])
X_test_multi = hstack([X_test_text, csr_matrix(X_test_vitals)])
# Train class-balanced Logistic Regression
baseline_clf = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
baseline_clf.fit(X_train_multi, train_df["high_acuity"])
y_pred_base = baseline_clf.predict(X_test_multi)
y_prob_base = baseline_clf.predict_proba(X_test_multi)[:, 1]
print("Model 1 (TF-IDF + Vitals Logistic Regression) trained successfully!")

Model 1 (TF-IDF + Vitals Logistic Regression) trained successfully!


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_fn(batch):
   return tokenizer(batch["clean_text"], padding="max_length", truncation=True, max_length=64)
train_ds = Dataset.from_pandas(train_df[["clean_text", "high_acuity"]].rename(columns={"high_acuity": "label"}))
test_ds = Dataset.from_pandas(test_df[["clean_text", "high_acuity"]].rename(columns={"high_acuity": "label"}))
train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
training_args = TrainingArguments(
   output_dir="./clinicalbert_triage",
   eval_strategy="epoch",
   learning_rate=2e-5,
   per_device_train_batch_size=32,
   per_device_eval_batch_size=64,
   num_train_epochs=2,
   weight_decay=0.01,
   logging_steps=50,
   report_to="none"
)
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=train_ds,
   eval_dataset=test_ds
)
trainer.train()
# Extract predicted probabilities from ClinicalBERT
preds_output = trainer.predict(test_ds)
y_prob_bert = torch.softmax(torch.tensor(preds_output.predictions), dim=1)[:, 1].numpy()
y_pred_bert = (y_prob_bert >= 0.5).astype(int)
print("Model 2 (Bio_ClinicalBERT) trained and evaluated successfully!")

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Epoch,Training Loss,Validation Loss
1,No log,0.089591
2,0.201636,0.030292


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model 2 (Bio_ClinicalBERT) trained and evaluated successfully!


In [6]:
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
def evaluate_clinical_safety(y_true, y_pred, y_prob, model_name):
   tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
   auroc = roc_auc_score(y_true, y_prob)
   macro_f1 = f1_score(y_true, y_pred, average="macro")
   # Critical Under-Triage Rate = False Negatives / Total Actual High-Acuity Cases
   under_triage_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0
   # Flag ambiguous cases where model confidence sits near the decision boundary (0.35 - 0.65)
   uncertain_cases = np.mean((y_prob >= 0.35) & (y_prob <= 0.65))
   print(f"--- {model_name} ---")
   print(f"AUROC:                     {auroc:.4f}")
   print(f"Macro-F1:                  {macro_f1:.4f}")
   print(f"Critical Under-Triage:     {under_triage_rate:.2%} (Missed ESI 1-2 emergencies)")
   print(f"Safety-Gate Escalations:   {uncertain_cases:.2%} (Deferred to human clinician)\n")
evaluate_clinical_safety(test_df["high_acuity"], y_pred_base, y_prob_base, "Model 1: TF-IDF + Vitals (Logistic Regression)")
evaluate_clinical_safety(test_df["high_acuity"], y_pred_bert, y_prob_bert, "Model 2: Bio_ClinicalBERT")

--- Model 1: TF-IDF + Vitals (Logistic Regression) ---
AUROC:                     1.0000
Macro-F1:                  1.0000
Critical Under-Triage:     0.00% (Missed ESI 1-2 emergencies)
Safety-Gate Escalations:   0.00% (Deferred to human clinician)

--- Model 2: Bio_ClinicalBERT ---
AUROC:                     1.0000
Macro-F1:                  1.0000
Critical Under-Triage:     0.00% (Missed ESI 1-2 emergencies)
Safety-Gate Escalations:   0.00% (Deferred to human clinician)

